In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("CNN Layer available :", layers.Conv2D)
print("TensorFlow version:", tf.__version__)


CNN Layer available : <class 'keras.src.layers.convolutional.conv2d.Conv2D'>
TensorFlow version: 2.21.0


In [2]:
IMG_WIDTH = 224
IMG_HEIGHT = 224
IMG_CHANNELS = 3
NUM_CLASSES = 4

def build_cnn(
        input_shape=(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS),
        num_classes=NUM_CLASSES,
):
    model = keras.Sequential(
        [
            keras.Input(shape=input_shape),
            layers.Conv2D(
                filters=32,
                kernel_size=(3, 3),
                padding="same",
                activation="relu"
            ),
            layers.MaxPooling2D(pool_size=(2, 2)),

            layers.Conv2D(
                filters=64,
                kernel_size=(3, 3),
                padding="same",
                activation="relu",
            ),

            layers.MaxPooling2D(pool_size=(2, 2)),

            layers.Conv2D(
                filters=128,
                kernel_size=(3, 3),
                padding="same",
                activation="relu",
            ),

            layers.MaxPooling2D(pool_size=(2, 2)),

            layers.GlobalAveragePooling2D(),
            layers.Dense(128, activation="relu"),
            layers.Dropout(0.4),
            layers.Dense(num_classes, activation="softmax"),
        ],
        name="brainscan_cnn",
    )
    return model

model = build_cnn()
print(len(model.layers))

10


E0000 00:00:1792183105.750445  271272 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [3]:
model.summary()

Model: "brainscan_cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 110,276 (430.77 KB)

 Trainable params: 110,276 (430.77 KB)

 Non-trainable params: 0 (0.00 B)

### Compiling the CNN model( using Adam Optimizer) & Hyperparameters



In [4]:
LEARNING_RATE = 0.001
EPOCHS = 30
BATCH_SIZE = 32

In [5]:
optimizer = keras.optimizers.Adam(
    learning_rate=LEARNING_RATE

)

model.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
print("model compiled successfully")

model compiled successfully


In [6]:
print("Loss function: ", model.loss)
print("Metrics configured: ", model.metrics_names)

Loss function:  sparse_categorical_crossentropy
Metrics configured:  ['loss', 'compile_metrics']


### plot_mode visuals

In [7]:
from pathlib import Path
from tensorflow.keras.utils import plot_model

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent
output_path = project_root / "models" / "02_cnn_architecture.png"

output_path.parent.mkdir(parents=True, exist_ok=True)
plot_model(
    model,
    to_file=str(output_path),
    show_shapes=True,
    show_layer_names=True,
    show_dtype=True,
    dpi=120,
)
print("visuals saved to: ", output_path)

visuals saved to:  /home/alaric/Desktop/studies/projects/DL/BrainScan/models/02_cnn_architecture.png


In [8]:

from pathlib import Path
import numpy as np

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

data_dir = project_root / "data" / "processed"

X_train = np.load(data_dir / "X_train.npy")
X_test = np.load(data_dir / "X_test.npy")
y_train = np.load(data_dir / "y_train.npy")
y_test = np.load(data_dir / "y_test.npy")

print("Training images:", X_train.shape)
print("Training labels:", y_train.shape)
print("Test images:", X_test.shape)
print("Test labels:", y_test.shape)

Training images: (5618, 224, 224, 3)
Training labels: (5618,)
Test images: (1405, 224, 224, 3)
Test labels: (1405,)


In [9]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = {
    int(class_id) : float(weight)
    for class_id, weight in zip(classes, weights)
}
print(class_weights)

{0: 1.0828835774865073, 1: 1.0672492401215805, 2: 0.8778125, 3: 0.999644128113879}


In [10]:
import time


start_time = time.time()

history = model.fit(
    X_train,
    y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    class_weight=class_weights,
    verbose=1,
)
training_duration = time.time() - start_time

print(f"Training duration: {training_duration / 60:.2f} minutes")

Epoch 1/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 135s 948ms/step - accuracy: 0.4230 - loss: 1.1949 - val_accuracy: 0.6023 - val_loss: 0.9655
Epoch 2/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 130s 922ms/step - accuracy: 0.6253 - loss: 0.9397 - val_accuracy: 0.6895 - val_loss: 0.8035
Epoch 3/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 130s 923ms/step - accuracy: 0.6722 - loss: 0.8436 - val_accuracy: 0.7002 - val_loss: 0.7571
Epoch 4/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 129s 918ms/step - accuracy: 0.7014 - loss: 0.7728 - val_accuracy: 0.7331 - val_loss: 0.6811
Epoch 5/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 129s 918ms/step - accuracy: 0.7134 - loss: 0.7497 - val_accuracy: 0.7180 - val_loss: 0.6880
Epoch 6/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 129s 916ms/step - accuracy: 0.7270 - loss: 0.6948 - val_accuracy: 0.7171 - val_loss: 0.7041
Epoch 7/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 129s 917ms/step - accuracy: 0.7432 - loss: 0.6862 - val_accuracy: 0.7037 - val_loss: 0.6998
Epoch 8/30
141/141 ━━━━━━━━━━━━━━━━━━━━ 129s 916ms/step - accuracy: 0.7592 -

In [10]:
from tensorflow.keras.callbacks import ModelCheckpoint

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

models_dir = project_root / "models"
models_dir.mkdir(parents=True, exist_ok=True)

best_model_path = models_dir / "brainscan_best.keras"
checkpoint = ModelCheckpoint(
    filepath=str(best_model_path),
    monitor="val_loss",
    save_best_only=True,
    mode="min",
    verbose=1
)